# 1단계 — 데이터 파이프라인 구축

**목표:** 366만 행의 원본 CSV를 분석 가능한 시간 단위 데이터로 정제·집계한다.

**입력:** `data/raw/` 폴더의 CSV 6개
- 미금역사거리.csv, 동막교사거리.csv, 오리삼거리.csv (원본 데이터)
- 차종구분.csv, 이동류구분.csv, 접근로정보.csv (코드북)

**출력:** `data/processed/hourly_panel.parquet`

**분석 단위:** 교차로 × 접근로 × 이동류 × 1시간

---

## 셀 1 — 라이브러리 및 경로 설정

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# notebooks/ 안에서 실행하므로 상위 폴더가 프로젝트 루트
BASE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW  = BASE / 'data' / 'raw'
PROC = BASE / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)

print('프로젝트 루트:', BASE)
print('원본 폴더 존재:', RAW.exists())
print('원본 파일 목록:', [f.name for f in RAW.glob('*.csv')])

프로젝트 루트: C:\Users\chpar\Exercise_CP\work\파란학기 응용
원본 폴더 존재: True
원본 파일 목록: ['동막교사거리.csv', '미금역사거리.csv', '오리삼거리.csv', '이동류구분.csv', '접근로정보.csv', '차종구분.csv']


> **확인:** CSV 파일 6개가 모두 출력되어야 한다. 안 보이면 경로부터 고칠 것.

---
## 셀 2 — 코드북 로드

숫자 코드를 사람이 읽을 수 있는 이름으로 바꾸기 위한 사전을 만든다.

In [2]:
car = pd.read_csv(RAW / '차종구분.csv')
mov = pd.read_csv(RAW / '이동류구분.csv')
ave = pd.read_csv(RAW / '접근로정보.csv')

car_map = dict(zip(car.CarModelType, car.CarModelName))   # 1 → 승용차
mov_map = dict(zip(mov.MovementType, mov.MovementName))   # 1 → 직진
ave_map = dict(zip(ave.AvenueSeq,    ave.AvenueName))     # 142 → 청송마을 사거리 방면

print('차종:', car_map)
print('이동류:', mov_map)
print('접근로:', len(ave_map), '개')
display(ave[['AvenueSeq', 'IntersectionSeq', 'AvenueName']])

차종: {1: '승용차', 2: '소형버스', 3: '대형버스', 4: '소형트럭', 5: '대형트럭', 6: '오토바이'}
이동류: {1: '직진', 2: '좌회전', 3: '우회전', 4: '유턴'}
접근로: 11 개


,AvenueSeq,IntersectionSeq,AvenueName
0,16,5,오리역 방면
1,17,5,오리교1사거리 방면
2,18,5,농수산물센터 방면
3,142,37,청송마을 사거리 방면
4,143,37,까치마을사거리 방면
5,144,37,미금역 방면
6,145,37,정자일로 방면
7,183,48,동막교삼거리 방면
8,184,48,오리공원 방면
9,185,48,오리역 방면


---
## 셀 3 — 원본 데이터 적재

dtype을 지정해 메모리를 절약한다 (약 223MB → 84MB). 로딩에 20~30초 소요.

In [3]:
INTERSECTIONS = {
    '미금역사거리': '미금역사거리.csv',
    '동막교사거리': '동막교사거리.csv',
    '오리삼거리':   '오리삼거리.csv',
}

DTYPES = {
    'IntersectionSeq': 'int16', 'AvenueSeq': 'int16', 'LaneNum': 'int8',
    'CarModelType': 'int8', 'MovementType': 'int8',
    'Volume': 'int32', 'Speed': 'float32',
}

frames = []
for name, fn in INTERSECTIONS.items():
    d = pd.read_csv(RAW / fn, dtype=DTYPES, parse_dates=['CollectedDate'])
    d['교차로'] = name
    frames.append(d)
    print(f'{name}: {len(d):,}행')

raw = pd.concat(frames, ignore_index=True)
raw['교차로'] = raw['교차로'].astype('category')

print(f'\n통합: {len(raw):,}행')
print(f'메모리: {raw.memory_usage(deep=True).sum()/1024**2:.0f} MB')
raw.head()

미금역사거리: 1,657,362행
동막교사거리: 1,104,658행
오리삼거리: 898,568행

통합: 3,660,588행
메모리: 84 MB


,IntersectionSeq,AvenueSeq,CollectedDate,LaneNum,CarModelType,MovementType,Volume,Speed,교차로
0,37,142,2023-11-07 16:00:00,1,1,2,86,11.9884,미금역사거리
1,37,142,2023-11-07 16:00:00,1,1,4,69,64.4058,미금역사거리
2,37,142,2023-11-07 16:00:00,1,2,2,3,10.0000,미금역사거리
3,37,142,2023-11-07 16:00:00,1,2,4,1,140.0000,미금역사거리
4,37,142,2023-11-07 16:00:00,1,3,2,8,8.6250,미금역사거리


> **예상 결과:** 미금역 1,657,362 / 동막교 1,104,658 / 오리삼거리 898,568 → 합계 3,660,588행, 약 84MB

---
## 셀 4 — 코드를 라벨로 변환

In [4]:
raw['차종']   = raw.CarModelType.map(car_map).astype('category')
raw['이동류'] = raw.MovementType.map(mov_map).astype('category')
raw['접근로'] = raw.AvenueSeq.map(ave_map).astype('category')

# 매핑 실패(결측)가 없어야 정상
missing = raw[['차종', '이동류', '접근로']].isna().sum()
print('매핑 실패 건수:')
print(missing)
assert missing.sum() == 0, '코드북에 없는 값이 있습니다 — 확인 필요'

raw[['교차로', '접근로', '이동류', '차종', 'Volume', 'Speed']].head()

매핑 실패 건수:
차종     0
이동류    0
접근로    0
dtype: int64


,교차로,접근로,이동류,차종,Volume,Speed
0,미금역사거리,청송마을 사거리 방면,좌회전,승용차,86,11.9884
1,미금역사거리,청송마을 사거리 방면,유턴,승용차,69,64.4058
2,미금역사거리,청송마을 사거리 방면,좌회전,소형버스,3,10.0000
3,미금역사거리,청송마을 사거리 방면,유턴,소형버스,1,140.0000
4,미금역사거리,청송마을 사거리 방면,좌회전,대형버스,8,8.6250


---
## 셀 5 — 시간 단위 집계 (핵심)

**속도는 반드시 교통량 가중평균으로 계산한다.**
단순평균을 쓰면 "1대가 100km/h로 지나간 기록"과 "300대가 20km/h로 지나간 기록"이
같은 무게로 계산되어 실제와 전혀 다른 값이 나온다.

In [5]:
# 가중평균을 위해 먼저 (속도 × 대수)를 만들어 합산
raw['속도x대수'] = raw.Speed.astype('float64') * raw.Volume

KEYS = ['교차로', 'AvenueSeq', '접근로', 'MovementType', '이동류', 'CollectedDate']

panel = (raw.groupby(KEYS, observed=True)
            .agg(교통량=('Volume', 'sum'),
                 속도합=('속도x대수', 'sum'))
            .reset_index())

# 대형차(대형버스=3, 대형트럭=5) 비율 — 6단계 변수 후보
heavy = (raw[raw.CarModelType.isin([3, 5])]
            .groupby(['교차로', 'AvenueSeq', 'MovementType', 'CollectedDate'], observed=True)
            .Volume.sum().rename('대형차량').reset_index())

panel = panel.merge(heavy, on=['교차로', 'AvenueSeq', 'MovementType', 'CollectedDate'], how='left')
panel['대형차량']   = panel['대형차량'].fillna(0).astype('int32')
panel['속도']       = (panel['속도합'] / panel['교통량']).astype('float32')
panel['대형차비율'] = (panel['대형차량'] / panel['교통량']).astype('float32')
panel = panel.drop(columns=['속도합'])

print('집계 결과:', panel.shape)
panel.head()

집계 결과: (531988, 10)


,교차로,AvenueSeq,접근로,MovementType,이동류,CollectedDate,교통량,대형차량,속도,대형차비율
0,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-07 16:00:00,1152,139,46.241318,0.120660
1,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-07 17:00:00,1243,118,42.220432,0.094932
2,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-07 18:00:00,1232,114,42.673710,0.092532
3,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-07 19:00:00,1354,152,44.280643,0.112260
4,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-07 20:00:00,988,154,47.227726,0.155870


---
## 셀 6 — 파생 변수 생성

In [6]:
panel['연']     = panel.CollectedDate.dt.year.astype('int16')
panel['월']     = panel.CollectedDate.dt.month.astype('int8')
panel['일']     = panel.CollectedDate.dt.day.astype('int8')
panel['시']     = panel.CollectedDate.dt.hour.astype('int8')
panel['요일']   = panel.CollectedDate.dt.dayofweek.astype('int8')   # 0=월 … 6=일
panel['요일명'] = panel.CollectedDate.dt.dayofweek.map(
    {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}).astype('category')
panel['평일']   = panel['요일'] < 5
panel['날짜']   = panel.CollectedDate.dt.normalize()

print(panel.shape)
panel.head()

(531988, 18)


,교차로,AvenueSeq,접근로,MovementType,이동류,CollectedDate,교통량,대형차량,속도,대형차비율,연,월,일,시,요일,요일명,평일,날짜
0,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-07 16:00:00,1152,139,46.241318,0.120660,2023,11,7,16,1,화,True,2023-11-07
1,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-07 17:00:00,1243,118,42.220432,0.094932,2023,11,7,17,1,화,True,2023-11-07
2,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-07 18:00:00,1232,114,42.673710,0.092532,2023,11,7,18,1,화,True,2023-11-07
3,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-07 19:00:00,1354,152,44.280643,0.112260,2023,11,7,19,1,화,True,2023-11-07
4,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-07 20:00:00,988,154,47.227726,0.155870,2023,11,7,20,1,화,True,2023-11-07


> **공휴일 플래그는 보류.** `holidays` 패키지가 필요하고, 4단계 이후에 추가해도 늦지 않다.
> 필요해지면: `pip install holidays` 후
> `panel['공휴일'] = panel.날짜.isin(holidays.KR(years=[2023, 2024, 2025]))`

---
## 셀 7 — 무결성 검증 (반드시 통과해야 함)

In [7]:
print('=== 무결성 검증 ===')

# ① 통행량 총합이 집계 전후로 같아야 함
before, after = raw.Volume.sum(), panel.교통량.sum()
print(f'원본 총 통행량 {before:,} / 집계 후 {after:,} → 일치: {before == after}')
assert before == after

# ② 값의 범위가 상식적인가
print(f'\n속도 범위: {panel.속도.min():.1f} ~ {panel.속도.max():.1f} km/h')
print(f'교통량 범위: {panel.교통량.min()} ~ {panel.교통량.max()} 대/시')
print(f'기간: {panel.CollectedDate.min()} ~ {panel.CollectedDate.max()}')

# ③ 결측 확인
na = panel.isna().sum()
print('\n결측치:')
print(na[na > 0] if na.sum() else '없음')

# ④ 교차로별 규모
print('\n교차로별 행수:')
print(panel.교차로.value_counts().to_string())

=== 무결성 검증 ===
원본 총 통행량 91,046,743 / 집계 후 91,046,743 → 일치: True

속도 범위: 0.0 ~ 140.0 km/h
교통량 범위: 1 ~ 1787 대/시
기간: 2023-11-07 16:00:00 ~ 2025-09-04 12:00:00

결측치:
없음

교차로별 행수:
교차로
미금역사거리    220524
동막교사거리    191135
오리삼거리     120329


> **예상 결과:** 총 통행량 91,046,743으로 일치 / 속도 0.0~140.0 / 교통량 1~1,787 / 결측 없음
>
> 속도 최댓값 140km/h는 도심 교차로에서 비현실적인 값으로, **2단계 품질 진단에서 다룰 항목**이다.

---
## 셀 8 — 시간 연속성 점검

센서가 꺼져 있던 구간이 있는지 확인한다. 결측이 **어느 시기에 몰려 있는지**가 특히 중요하다.

In [8]:
print('=== 교차로별 시간 결측 ===')
gap_summary = {}

for name in INTERSECTIONS:
    sub  = panel[panel.교차로 == name]
    full = pd.date_range(sub.CollectedDate.min(), sub.CollectedDate.max(), freq='h')
    have = pd.DatetimeIndex(sorted(sub.CollectedDate.unique()))
    miss = full.difference(have)
    gap_summary[name] = miss
    print(f'{name}: 결측 {len(miss):,} / 전체 {len(full):,} 시간 ({100*len(miss)/len(full):.2f}%)')

# 결측이 특정 시기에 몰려 있는지 확인
print('\n=== 결측이 많은 교차로의 월별 분포 ===')
for name, miss in gap_summary.items():
    if len(miss) > 100:
        by_month = pd.Series(1, index=miss).resample('ME').sum()
        print(f'\n[{name}]')
        print(by_month[by_month > 0].to_string())
        diffs = pd.Series(miss).diff().dt.total_seconds().div(3600)
        print(f'연속 결측(1시간 간격) 비율: {100*(diffs == 1).mean():.1f}%')

=== 교차로별 시간 결측 ===
미금역사거리: 결측 4 / 전체 16,005 시간 (0.02%)
동막교사거리: 결측 2,533 / 전체 16,005 시간 (15.83%)
오리삼거리: 결측 4 / 전체 16,005 시간 (0.02%)

=== 결측이 많은 교차로의 월별 분포 ===

[동막교사거리]
2024-04-30      4
2024-09-30    128
2024-10-31    242
2024-11-30    253
2024-12-31    744
2025-01-31    744
2025-02-28    418
연속 결측(1시간 간격) 비율: 96.7%


> **주목할 결과:**
> 미금역·오리삼거리는 결측이 0.02%로 사실상 완전하지만,
> **동막교사거리는 15.83%(2,533시간)** 가 비어 있다.
> 게다가 이 결측은 흩어져 있지 않고 **2024년 10월 ~ 2025년 2월에 집중된 연속 구간(96.7%)** 이다.
>
> 센서 장애나 공사 등 특정 사건을 시사하며,
> **2단계 품질 진단의 핵심 소재이자, 5단계에서 학습·검증 기간을 나눌 때 반드시 고려해야 할 제약**이다.

---
## 셀 9 — 저장

In [9]:
out = PROC / 'hourly_panel.parquet'
panel.to_parquet(out, index=False)

print(f'저장 완료: {out}')
print(f'파일 크기: {out.stat().st_size/1024**2:.1f} MB')
print(f'행 수: {len(panel):,}')

# 불러오기 확인
check = pd.read_parquet(out)
print('재로드 검증:', check.shape, '— 정상' if check.shape == panel.shape else '— 오류')

저장 완료: C:\Users\chpar\Exercise_CP\work\파란학기 응용\data\processed\hourly_panel.parquet
파일 크기: 5.6 MB
행 수: 531,988
재로드 검증: (531988, 18) — 정상


> **원본 160MB → 집계 후 약 5.6MB.**
> 앞으로 모든 단계는 이 파일 하나만 읽으면 되므로 매번 CSV를 다시 처리할 필요가 없다 (30초 → 1초 미만).

---
## 셀 10 — 결과 미리보기

In [10]:
print('=== 최종 데이터 구조 ===')
print(panel.dtypes.to_string())

print('\n=== 샘플 ===')
display(panel.head(10))

print('\n=== 교차로 × 접근로 조합 ===')
display(panel.groupby(['교차로', '접근로'], observed=True)
             .agg(행수=('교통량', 'size'),
                  평균교통량=('교통량', 'mean'),
                  평균속도=('속도', 'mean'))
             .round(1))

=== 최종 데이터 구조 ===
교차로                    category
AvenueSeq                 int16
접근로                    category
MovementType               int8
이동류                    category
CollectedDate    datetime64[ns]
교통량                       int32
대형차량                      int32
속도                      float32
대형차비율                   float32
연                         int16
월                          int8
일                          int8
시                          int8
요일                         int8
요일명                    category
평일                         bool
날짜               datetime64[ns]

=== 샘플 ===


,교차로,AvenueSeq,접근로,MovementType,이동류,CollectedDate,교통량,대형차량,속도,대형차비율,연,월,일,시,요일,요일명,평일,날짜
0,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-07 16:00:00,1152,139,46.241318,0.120660,2023,11,7,16,1,화,True,2023-11-07
1,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-07 17:00:00,1243,118,42.220432,0.094932,2023,11,7,17,1,화,True,2023-11-07
2,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-07 18:00:00,1232,114,42.673710,0.092532,2023,11,7,18,1,화,True,2023-11-07
3,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-07 19:00:00,1354,152,44.280643,0.112260,2023,11,7,19,1,화,True,2023-11-07
4,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-07 20:00:00,988,154,47.227726,0.155870,2023,11,7,20,1,화,True,2023-11-07
5,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-07 21:00:00,840,126,49.172611,0.150000,2023,11,7,21,1,화,True,2023-11-07
6,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-07 22:00:00,802,124,49.475060,0.154613,2023,11,7,22,1,화,True,2023-11-07
7,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-07 23:00:00,433,105,56.374138,0.242494,2023,11,7,23,1,화,True,2023-11-07
8,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-08 00:00:00,307,51,62.654720,0.166124,2023,11,8,0,2,수,True,2023-11-08
9,동막교사거리,183,동막교삼거리 방면,1,직진,2023-11-08 01:00:00,171,7,64.286545,0.040936,2023,11,8,1,2,수,True,2023-11-08



=== 교차로 × 접근로 조합 ===


행수  평균교통량       평균속도
교차로    접근로                                 
동막교사거리 경기고속삼거리 방면   49493   67.8  34.700001
       동막교삼거리 방면    49940  235.9  36.000000
       오리공원 방면      40524   47.8  29.600000
       오리역 방면       51178  181.1  31.100000
미금역사거리 까치마을사거리 방면   48117  112.6  53.900002
       미금역 방면       62400  207.2  19.700001
       정자일로 방면      47984  219.1  11.500000
       청송마을 사거리 방면  62023  178.6  28.200001
오리삼거리  농수산물센터 방면    46599  244.7  18.000000
       오리교1사거리 방면   45160   77.7  18.600000
       오리역 방면       28570  344.8  17.400000

---
# 1단계 완료

| 항목 | 값 |
|---|---|
| 입력 | CSV 6개, 약 160MB, 3,660,588행 |
| 출력 | `hourly_panel.parquet`, 약 5.6MB, 531,988행 |
| 소요 시간 | 약 30초 |
| 컬럼 | 교차로, 접근로, 이동류, 시각, 교통량, 속도, 대형차비율, 시간 파생변수 |

### 이 단계에서 확보한 발견 (2단계에서 다룰 것)

1. **동막교사거리 결측 15.83%** — 2024년 10월~2025년 2월 집중, 연속 구간
2. **속도 최댓값 140km/h** — 도심 교차로에서 비현실적
3. **교통량 최댓값 1,787대/시** — 접근로 단위 값의 타당성 검토 필요

---

### 문제가 생기면

| 증상 | 원인과 해결 |
|---|---|
| `FileNotFoundError` | 셀 1의 경로 출력 확인. `data/raw/`에 CSV가 있는지, 파일명이 정확한지 점검 |
| 한글 깨짐 | `pd.read_csv(..., encoding='utf-8-sig')` 또는 `encoding='cp949'` 추가 |
| `assert` 실패 (매핑) | 코드북에 없는 코드값 존재. `raw[raw.차종.isna()].CarModelType.unique()`로 확인 |
| 메모리 부족 | 셀 3에서 교차로를 하나씩 처리해 개별 저장 후 합치기 |